# Investigation 1 — Adjacency Signal (Q1)

**Hypothesis:** Companies that hold an active lease on a block adjacent to an available block bid on that available block at a meaningfully higher rate than companies without an adjacent position.

**Method:** For each (company, available-block) pair, check whether the company holds an edge-adjacent lease. Compare bid rates.

**Success:** Lift >= 3× with p < 0.05.

---
*Prototype using Sale 247 (March 2017). Swap to Dec 2025 for final validation.*

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from libpysal.weights import Rook

ROOT = os.path.abspath(os.path.join("..", ".."))
sys.path.insert(0, os.path.join(ROOT, "mvp0"))
import boem_loader as bl

SALE_DATE = pd.Timestamp("2017-03-22")
print(f"Reference sale date: {SALE_DATE.date()}")

## 1. Load data

In [ ]:
sales = bl.load_master_sales(os.path.join(ROOT, "data/lease-sales/master_lease_sales.csv"))
lh    = bl.load_lease_history(os.path.join(ROOT, "data/lease-sales/cleaned_lease_history.csv"))
lo    = bl.load_lease_owners(os.path.join(ROOT, "data/lease-sales/lseowndelimit.txt"))
blocks = bl.load_blocks(os.path.join(ROOT, "data/shapefiles/blocks.shp"), to_utm=True)

print(f"Sale bids: {len(sales)} rows, {sales['Lease_Number'].nunique()} unique blocks")
print(f"Lease history: {len(lh)} rows")
print(f"Lease owners: {len(lo)} rows")
print(f"Blocks: {len(blocks)} rows")

## 2. Identify the sale's universe of blocks

Limit to protraction areas where Sale 247 offered tracts. The "available" set is all blocks in those areas minus blocks with active leases.

In [ ]:
# Protraction areas in this sale
sale_prots = set(sales["Protraction_ID"].unique())
print(f"Sale protraction areas: {len(sale_prots)}")

# All blocks in those protraction areas
universe = blocks[blocks["Protraction_ID"].isin(sale_prots)].copy()
universe = universe.reset_index(drop=True)
# Give each block a unique integer index for the adjacency matrix
universe["block_idx"] = range(len(universe))
print(f"Blocks in sale universe: {len(universe)}")

## 3. Determine active leases at sale time

A lease was active as of the sale date if:
- It is still currently active (PRIMRY, PROD, UNIT, …), **or**
- It was terminated *after* the sale date (EXPIR, RELINQ, TERMED with `Status_Date > SALE_DATE`)

In [ ]:
active_statuses = ["PRIMRY", "PROD", "SOO", "SOP", "UNIT", "DSO", "OPERNS"]
terminal_statuses = ["EXPIR", "RELINQ", "TERMED", "CANCEL"]

active_mask = (
    lh["Lease_Status"].isin(active_statuses) |
    (lh["Lease_Status"].isin(terminal_statuses) & (lh["Status_Date"] > SALE_DATE))
)
active_leases = lh[active_mask].copy()
print(f"Leases active at sale time: {len(active_leases)}")
print(active_leases["Lease_Status"].value_counts().to_string())

## 4. Map active leases → (company, block)

Join lease history to lease owners (via lease number) to get the company behind each active lease.

In [ ]:
# Normalize lease numbers: lease_history uses '0035930', owners uses 'G35930'
active_leases["Lease_G"] = "G" + active_leases["Lease_Number"].str.lstrip("0")

# Get latest owner per lease as of sale date
lo_at_sale = lo[
    (lo["Asgn_Eff_Date"] <= SALE_DATE) &
    (lo["Owner_Aliquot"] == "1")  # primary aliquot only
].copy()
lo_at_sale = lo_at_sale.sort_values("Asgn_Eff_Date").drop_duplicates("Lease_Number", keep="last")

# Join
active_co = active_leases.merge(
    lo_at_sale[["Lease_Number", "Company_Number"]],
    left_on="Lease_G", right_on="Lease_Number",
    how="inner", suffixes=("_lh", "_lo"),
)
print(f"Active leases matched to owners: {len(active_co)}")

# Join to blocks (via Area_Code + Block_Number)
active_blocks = active_co.merge(
    universe[["AREA_CODE", "Block_Number", "block_idx"]],
    left_on=["Area_Code", "Block_Number"],
    right_on=["AREA_CODE", "Block_Number"],
    how="inner",
)
print(f"Active leases on blocks in sale universe: {len(active_blocks)}")
print(f"Unique companies with active leases in area: {active_blocks['Company_Number'].nunique()}")

## 5. Build block adjacency (Rook contiguity)

Two blocks are adjacent if they share a full edge (Rook contiguity — not diagonal).

In [ ]:
%%time
w = Rook.from_dataframe(universe, use_index=False)
print(f"Adjacency weights built: {w.n} blocks, mean {w.mean_neighbors:.1f} neighbors")

# Build a fast lookup: block_idx → set of neighbor block_idxs
adj = {i: set(w.neighbors[i]) for i in range(w.n)}

## 6. Build the analysis matrix

For each company that bid in Sale 247, check every block in the universe:
- `adjacent`: does this company hold a lease on any edge-adjacent block?
- `did_bid`: did the company bid on this block?

In [ ]:
# Companies that bid in Sale 247
bid_companies = sales["Company_Number"].unique()
print(f"Companies that bid: {len(bid_companies)}")

# For each company, the set of block_idxs they hold leases on
company_held = (
    active_blocks.groupby("Company_Number")["block_idx"]
    .apply(set)
    .to_dict()
)

# Bid lookup: (Company_Number, Protraction_ID, Block_Number) → True
bid_set = set(
    zip(sales["Company_Number"], sales["Protraction_ID"], sales["Block_Number"])
)

# Block info for lookup
blk_info = universe[["block_idx", "Protraction_ID", "Block_Number"]].to_dict("records")

# Build rows
rows = []
for co in bid_companies:
    held = company_held.get(co, set())
    # Blocks adjacent to any held block
    adj_blocks = set()
    for h in held:
        adj_blocks.update(adj.get(h, set()))
    adj_blocks -= held  # exclude blocks the company already holds

    for blk in blk_info:
        idx = blk["block_idx"]
        if idx in held:
            continue  # skip blocks company already leases
        is_adj = idx in adj_blocks
        did_bid = (co, blk["Protraction_ID"], blk["Block_Number"]) in bid_set
        rows.append((co, idx, is_adj, did_bid))

matrix = pd.DataFrame(rows, columns=["Company", "block_idx", "adjacent", "did_bid"])
print(f"Analysis matrix: {len(matrix):,} (company × block) pairs")
print(f"  adjacent=True: {matrix['adjacent'].sum():,}")
print(f"  did_bid=True:  {matrix['did_bid'].sum():,}")

## 7. Compute bid rates and lift

In [ ]:
ct = pd.crosstab(matrix["adjacent"], matrix["did_bid"], margins=True)
ct.index = ct.index.map({False: "No adjacency", True: "Has adjacency", "All": "All"})
ct.columns = ct.columns.map({False: "No bid", True: "Bid", "All": "Total"})
print("=== Contingency table ===")
display(ct)

# Bid rates
adj_yes = matrix[matrix["adjacent"]]
adj_no  = matrix[~matrix["adjacent"]]

rate_adj = adj_yes["did_bid"].mean()
rate_non = adj_no["did_bid"].mean()
lift = rate_adj / rate_non if rate_non > 0 else float("inf")

print(f"\nBid rate (adjacent):     {rate_adj:.4%}  ({adj_yes['did_bid'].sum()} / {len(adj_yes)})")
print(f"Bid rate (non-adjacent): {rate_non:.4%}  ({adj_no['did_bid'].sum()} / {len(adj_no)})")
print(f"Lift: {lift:.1f}×")

## 8. Chi-square test

In [ ]:
ct_vals = pd.crosstab(matrix["adjacent"], matrix["did_bid"])
chi2, p_value, dof, expected = chi2_contingency(ct_vals)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value:              {p_value:.2e}")
print(f"Degrees of freedom:   {dof}")

if p_value < 0.05:
    print("\n→ Statistically significant (p < 0.05)")
else:
    print("\n→ NOT statistically significant (p >= 0.05)")

## 9. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: bid rates
ax = axes[0]
rates = pd.Series({"Adjacent": rate_adj * 100, "Non-adjacent": rate_non * 100})
rates.plot.bar(ax=ax, color=["#2196F3", "#9E9E9E"], edgecolor="black")
ax.set_ylabel("Bid rate (%)")
ax.set_title(f"Adjacency Lift: {lift:.1f}×  (p = {p_value:.2e})")
ax.tick_params(axis="x", rotation=0)
for i, v in enumerate(rates):
    ax.text(i, v + 0.1, f"{v:.2f}%", ha="center", fontsize=10)

# Map: blocks colored by adjacency status
from matplotlib.patches import Patch

ax = axes[1]
universe.plot(ax=ax, color="whitesmoke", edgecolor="gray", linewidth=0.2)

# Highlight blocks with active leases
held_idxs = set()
for s in company_held.values():
    held_idxs.update(s)
held_blocks = universe[universe["block_idx"].isin(held_idxs)]
held_blocks.plot(ax=ax, color="#FFC107", edgecolor="gray", linewidth=0.3)

# Highlight bid blocks
bid_merged = sales.merge(
    universe[["Protraction_ID", "Block_Number", "geometry"]],
    on=["Protraction_ID", "Block_Number"], how="inner",
)
bid_gdf = gpd.GeoDataFrame(bid_merged, geometry="geometry")
bid_gdf.plot(ax=ax, color="#F44336", edgecolor="black", linewidth=0.5)

ax.set_title("Sale 247 — Active Leases & Bids")

# Create manual legend to avoid PatchCollection warning
legend_elements = [
    Patch(facecolor="#FFC107", edgecolor="gray", label="Active lease"),
    Patch(facecolor="#F44336", edgecolor="black", label="Bid placed"),
]
ax.legend(handles=legend_elements, loc="lower left", fontsize=8)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()

## 10. Interpretation

| Result | Interpretation | Action |
|--------|---------------|--------|
| Lift >= 3×, p < 0.05 | Strong signal. Adjacency is a real predictor. | Proceed to MVP 1 as planned. |
| Lift 1.5×–3×, p < 0.05 | Moderate signal. Useful but not dominant. | Proceed, but weight adjacency appropriately. |
| Lift < 1.5× or p > 0.05 | Weak or noisy signal. | Do not proceed to MVP 1 as written. |

**Note:** This prototype uses Sale 247 with an approximate reconstruction of active leases.
For the final validation, replace with Dec 2025 sale data and a verified lease snapshot.